# 02 - Governed SQL Agent With LangGraph

This example shows the agent pattern Metatate is meant to support:

User asks a business question, the agent discovers context, drafts SQL, validates it through Metatate, and revises or blocks the output based on the decision.

If `langgraph` is installed, the cells build a small graph. Without it, the notebook runs the same steps as plain Python so the example stays readable.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


In [ ]:
from typing import TypedDict, Any

TABLE = "ACMECLOUD_DEMO.PUBLIC.CUSTOMERS"
QUESTION = "Show ARR by region for active customers. Include email if it helps identify accounts."


class AgentState(TypedDict, total=False):
    question: str
    context: dict[str, Any]
    draft_sql: str
    validation: dict[str, Any]
    final_sql: str
    decision: str
    notes: list[str]


def discover_context_step(state: AgentState) -> AgentState:
    context = client.get_decision_context(TABLE)
    return {**state, "context": context}


def draft_sql_step(state: AgentState) -> AgentState:
    # The initial draft intentionally includes EMAIL so Metatate can catch it.
    draft_sql = (
        "SELECT region, email, SUM(arr) AS arr "
        "FROM ACMECLOUD_DEMO.PUBLIC.CUSTOMERS "
        "WHERE account_status = 'active' "
        "GROUP BY region, email"
    )
    return {**state, "draft_sql": draft_sql}


def validate_sql_step(state: AgentState) -> AgentState:
    validation = client.validate_query_context(
        state["draft_sql"],
        operation="read",
        intended_use="analytics",
        actor_role="DATA_ANALYST",
    )
    decision = validation["data"].get("decision", {}).get("decision", "UNKNOWN")
    return {**state, "validation": validation, "decision": decision}


def revise_sql_step(state: AgentState) -> AgentState:
    findings = state["validation"]["data"].get("sql_findings", [])
    final_sql = state["draft_sql"]
    notes = []
    if any(item.get("code") == "PII_COLUMN_SELECTED" for item in findings):
        final_sql = (
            "SELECT region, SUM(arr) AS arr "
            "FROM ACMECLOUD_DEMO.PUBLIC.CUSTOMERS "
            "WHERE account_status = 'active' "
            "GROUP BY region"
        )
        notes.append("Removed EMAIL because Metatate flagged it as PII for analytics.")
    return {**state, "final_sql": final_sql, "notes": notes}


In [ ]:
try:
    from langgraph.graph import END, StateGraph

    workflow = StateGraph(AgentState)
    workflow.add_node("discover_context", discover_context_step)
    workflow.add_node("draft_sql", draft_sql_step)
    workflow.add_node("validate_sql", validate_sql_step)
    workflow.add_node("revise_sql", revise_sql_step)
    workflow.set_entry_point("discover_context")
    workflow.add_edge("discover_context", "draft_sql")
    workflow.add_edge("draft_sql", "validate_sql")
    workflow.add_edge("validate_sql", "revise_sql")
    workflow.add_edge("revise_sql", END)
    app = workflow.compile()
    result = app.invoke({"question": QUESTION})
    print("Ran with LangGraph.")
except ImportError:
    result = {"question": QUESTION}
    for step in (discover_context_step, draft_sql_step, validate_sql_step, revise_sql_step):
        result = step(result)
    print("LangGraph is not installed. Ran the same flow as plain Python.")

print("Question:")
print(result["question"])
print("\nDraft SQL:")
print(result["draft_sql"])
print("\nMetatate decision:")
print(result["decision"])
print("\nFinal SQL:")
print(result["final_sql"])
print("\nNotes:")
print("\n".join(result.get("notes", [])))


The point is not the SQL itself. The point is that the agent does not decide alone. It checks the decision layer before handing SQL to a user or workflow.
